In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga de los ficheros de datos parciales

Se cargan todos los ficheros de servicios que se han ido creando en los notebooks parciales

In [2]:
def load_with_cusec(path):
    df = pd.read_csv(
        path,
        encoding="utf-8-sig",
        sep=";",          # column separator
        decimal=",",      # European decimal format
        dtype=str         # force all fields to string to avoid type issues
    )
    
    # Normalize CUSEC if present
    if "CUSEC" in df.columns:
        df["CUSEC"] = (
            df["CUSEC"]
            .astype(str)
            .str.strip()
            .str.zfill(10)
        )

    # Normalize CMuni if present
    if "CMuni" in df.columns:
        df["CMuni"] = (
            df["CMuni"]
            .astype(str)
            .str.strip()
            .str.zfill(5)
        )
    
    # Numeric normalization only if columns exist
    numeric_cols = ["Población", "area_km2"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(",", ".", regex=False)  # support decimal comma
                .str.strip()
                .replace("", None)
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    return df
    
# Por sección censal
demograficos_seccion = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "indicadores_demográficos_por_seccion.csv"))
edades_seccion = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "indicadores_edades_por_seccion.csv"))
sexos_seccion = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "indicadores_sexo_por_seccion.csv"))
superficies_seccion = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "superficie_seccion.csv"))

# Por municipio
demograficos_municipio = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "indicadores_demográficos_por_municipio.csv"))
edades_municipio = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "indicadores_edades_por_municipio.csv"))
sexos_municipio = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "indicadores_sexo_por_municipio.csv"))
superficies_municipio = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "superficie_municipio.csv"))

# Por provincia
demograficos_provincia = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "indicadores_demográficos_por_provincia.csv"))
edades_provincia = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "indicadores_edades_por_provincia.csv"))
sexos_provincia = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "indicadores_sexo_por_provincia.csv"))
superficies_provincia = load_with_cusec(os.path.join(DATA_OUTPUTS_DD, "superficie_provincia.csv"))

# Cargo el shp para obtener el id:
ruta_shp_secciones = os.path.join(DATA_OUTPUTS_DIR, "Shapefiles", "cyl_2022.shp")
gdf_secciones = gpd.read_file(ruta_shp_secciones)
gdf_secciones["CUSEC"] = (
    gdf_secciones["CUSEC"]
    .astype(str)
    .str.strip()
    .str.zfill(10)  # rellena con ceros a la izquierda hasta 10 caracteres
)

In [3]:
# Uno los datos por Provincia, CUSEC, CMuni y Periodo
DD_seccion = (
    demograficos_seccion
    .merge(edades_seccion, on=["Provincia","CUSEC","CMuni","Periodo"], how="inner")
    .merge(sexos_seccion, on=["Provincia", "CUSEC","CMuni", "Periodo"], how="inner")
    .merge(
        gdf_secciones.drop(columns=["geometry","NMUN","NPRO"]),
        on="CUSEC",
        how="left"
    )    
)

# Integro la superficie de la sección censal y obtengo densidad de población
DD_seccion = DD_seccion.merge(
    superficies_seccion[["Provincia", "CMuni", "CUSEC", "area_km2"]],
    on=["Provincia", "CMuni", "CUSEC"],
    how="left"
)

DD_seccion["Densidad de población"] = DD_seccion["Población"] / DD_seccion["area_km2"]

# Compruebo que se haya cargado
print(DD_seccion.info())
DD_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10571 entries, 0 to 10570
Data columns (total 43 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Provincia                            10571 non-null  object 
 1   CMuni                                10571 non-null  object 
 2   CUSEC                                10571 non-null  object 
 3   Periodo                              10571 non-null  object 
 4   Edad_media_de_la_población           10571 non-null  object 
 5   Porcentaje_de_hogares_unipersonales  10571 non-null  object 
 6   Porcentaje_de_población_española     10571 non-null  object 
 7   Tamaño_medio_del_hogar               10571 non-null  object 
 8   Porcentaje_de_población_extranjera   10571 non-null  object 
 9   100_y_más_años                       10571 non-null  object 
 10  De_0_a_4_años                        10571 non-null  object 
 11  De_10_a_14_años             

,Provincia,CMuni,CUSEC,Periodo,Edad_media_de_la_población,Porcentaje_de_hogares_unipersonales,Porcentaje_de_población_española,Tamaño_medio_del_hogar,Porcentaje_de_población_extranjera,100_y_más_años,...,Indice_Envejecimiento,Tasa_Dependencia,Hombres,Mujeres,Población,%_Hombres,%_Mujeres,Seccion_id,area_km2,Densidad de población
5681,Salamanca,37263,3726301001,2021,"55,300","50,000","98,300","1,800","1,700",0,...,"6,960","0,980",176,193,369,"47,700","52,300",1846.0,10.392,35.508083
4145,Palencia,34045,3404501001,2022,"58,700","43,800","100,000","2,200","0,000",0,...,"8,670","0,720",39,30,69,"56,520","43,480",1337.0,18.266,3.777510
3294,Leon,24089,2408909001,2022,"46,600","37,900","90,600","2,200","9,400",0,...,"2,060","0,620",948,1003,1951,"48,590","51,410",1073.0,8.164,238.975992
1153,Burgos,09059,0905902001,2022,"46,700","41,900","91,100","2,100","8,900",0,...,"1,900","0,540",338,431,769,"43,950","56,050",383.0,0.064,12015.625000
2382,Burgos,09366,0936601001,2023,"60,700","32,400","92,300","2,100","7,700",1,...,NaN,"0,800",45,29,74,"60,810","39,190",796.0,14.638,5.055335


In [4]:
# Uno los datos por Provincia, CMuni y Periodo
DD_municipio = (
    demograficos_municipio
    .merge(edades_municipio, on=["Provincia","CMuni","Periodo"], how="inner")
    .merge(sexos_municipio, on=["Provincia","CMuni", "Periodo"], how="inner")
)

# Integro la superficie de la sección censal y obtengo densidad de población
DD_municipio = DD_municipio.merge(
    superficies_municipio[["Provincia", "CMuni", "area_km2"]],
    on=["Provincia", "CMuni"],
    how="left"
)

DD_municipio["Densidad de población"] = DD_municipio["Población"] / DD_municipio["area_km2"]

# Compruebo que se haya cargado
print(DD_municipio.info())
DD_municipio.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6744 entries, 0 to 6743
Data columns (total 41 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Provincia                            6744 non-null   object 
 1   CMuni                                6744 non-null   object 
 2   Periodo                              6744 non-null   object 
 3   Edad_media_de_la_población           6744 non-null   object 
 4   Porcentaje_de_hogares_unipersonales  6744 non-null   object 
 5   Porcentaje_de_población_española     6744 non-null   object 
 6   Porcentaje_de_población_extranjera   6744 non-null   object 
 7   Tamaño_medio_del_hogar               6744 non-null   object 
 8   100_y_más_años                       6744 non-null   object 
 9   De_0_a_4_años                        6744 non-null   object 
 10  De_5_a_9_años                        6744 non-null   object 
 11  De_10_a_14_años               

,Provincia,CMuni,Periodo,Edad_media_de_la_población,Porcentaje_de_hogares_unipersonales,Porcentaje_de_población_española,Porcentaje_de_población_extranjera,Tamaño_medio_del_hogar,100_y_más_años,De_0_a_4_años,...,poblacion_mayor_65,Indice_Envejecimiento,Tasa_Dependencia,Hombres,Mujeres,Población,%_Hombres,%_Mujeres,area_km2,Densidad de población
1267,Burgos,09226,2022,"66,500","67,500","94,800","5,200","1,400",0,0,...,33,inf,"1,435",34,22,56,"60,714","39,286",15.100,3.708609
4776,Soria,42001,2021,"53,600","45,000","89,000","11,000","2,100",0,6,...,99,"5,824","0,753",142,128,270,"52,593","47,407",23.413,11.532055
2718,Palencia,34098,2021,"43,400","33,800","92,900","7,100","2,300",0,43,...,153,"1,070","0,407",534,489,1023,"52,199","47,801",27.832,36.756252
825,Burgos,09035,2021,"51,700","43,700","91,000","9,000","2,300",0,8,...,108,"3,176","0,785",191,132,323,"59,133","40,867",36.494,8.850770
1886,Leon,24010,2023,"46,255","34,512","89,723","10,277","2,280",5,356,...,2461,"1,808","0,607",4841,5274,10115,"47,860","52,140",19.727,512.749024


In [5]:
# Uno los datos por Provincia y Periodo
DD_provincia = (
    demograficos_provincia
    .merge(edades_provincia, on=["Provincia","Periodo"], how="inner")
    .merge(sexos_provincia, on=["Provincia","Periodo"], how="inner")
)

# Integro la superficie de la sección censal y obtengo densidad de población
DD_provincia = DD_provincia.merge(
    superficies_provincia[["Provincia", "area_km2"]],
    on=["Provincia"],
    how="left"
)

DD_provincia["Densidad de población"] = DD_provincia["Población"] / DD_provincia["area_km2"]

# Compruebo que se haya cargado
print(DD_provincia.info())
DD_provincia.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 40 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Provincia                            27 non-null     object 
 1   Periodo                              27 non-null     object 
 2   Edad_media_de_la_población           27 non-null     object 
 3   Porcentaje_de_hogares_unipersonales  27 non-null     object 
 4   Porcentaje_de_población_española     27 non-null     object 
 5   Porcentaje_de_población_extranjera   27 non-null     object 
 6   Tamaño_medio_del_hogar               27 non-null     object 
 7   100_y_más_años                       27 non-null     object 
 8   De_0_a_4_años                        27 non-null     object 
 9   De_5_a_9_años                        27 non-null     object 
 10  De_10_a_14_años                      27 non-null     object 
 11  De_15_a_19_años                   

,Provincia,Periodo,Edad_media_de_la_población,Porcentaje_de_hogares_unipersonales,Porcentaje_de_población_española,Porcentaje_de_población_extranjera,Tamaño_medio_del_hogar,100_y_más_años,De_0_a_4_años,De_5_a_9_años,...,poblacion_mayor_65,Indice_Envejecimiento,Tasa_Dependencia,Hombres,Mujeres,Población,%_Hombres,%_Mujeres,area_km2,Densidad de población
13,Salamanca,2022,"53,251","41,031","95,431","4,569","2,099",266,9704,12047,...,88888,"2,302","0,641",158487,167798,326285,"48,573","51,427",12359.830,26.398826
25,Zamora,2022,"54,398","40,873","95,816","4,184","2,074",133,4188,5147,...,53016,"3,222","0,706",83170,84698,167868,"49,545","50,455",10569.189,15.882770
1,Avila,2022,"52,431","42,398","93,945","6,055","2,091",112,4839,6315,...,41873,"2,121","0,632",80060,79042,159102,"50,320","49,680",8048.839,19.767074
19,Soria,2022,"52,659","44,006","91,644","8,356","2,129",83,3068,3507,...,22548,"1,992","0,622",44754,43576,88330,"50,667","49,333",10294.970,8.579918
15,Segovia,2021,"48,879","37,581","88,206","11,794","2,284",83,5581,6705,...,34600,"1,636","0,570",77152,76474,153626,"50,221","49,779",6791.382,22.620727


In [6]:
# Añado si es rural o no por los criterios de Eurostat
DD_municipio["Rural"] = (
    (DD_municipio["Densidad de población"] < 150))
# Propago a secciones
df_rural = DD_municipio[["CMuni", "Periodo", "Rural"]].copy()

df_rural["CMuni"] = df_rural["CMuni"].astype(str).str.zfill(5)
DD_seccion = DD_seccion.merge(
    df_rural,
    on=["CMuni", "Periodo"],
    how="inner"
)


In [7]:
cols = ['Población', 'Densidad de población', 'Rural']

# --- Sections ---
rural_sections = DD_seccion[DD_seccion['Rural'] == True][cols].sample(3)
non_rural_sections = DD_seccion[DD_seccion['Rural'] == False][cols].sample(3)

print(f"Rural sections:\n{rural_sections}\n")
print(f"Non-rural sections:\n{non_rural_sections}\n")

# --- Municipalities ---
rural_munis = DD_municipio[DD_municipio['Rural'] == True][cols].sample(3)
non_rural_munis = DD_municipio[DD_municipio['Rural'] == False][cols].sample(3)

print(f"Rural municipalities:\n{rural_munis}\n")
print(f"Non-rural municipalities:\n{non_rural_munis}\n")

Rural sections:
      Población  Densidad de población  Rural
8318       1532            9456.790123   True
7253        104               3.947019   True
866         106               9.065253   True

Non-rural sections:
      Población  Densidad de población  Rural
2007       1010           31562.500000  False
6784       2220             555.555556  False
5957       1032           21957.446809  False

Rural municipalities:
      Población  Densidad de población  Rural
5880        120               5.299885   True
2115       1706              48.242513   True
3192         33               1.779359   True

Non-rural municipalities:
      Población  Densidad de población  Rural
886      173197            1615.779310  False
5354      21950            1873.826191  False
2998       6479             161.975000  False



In [8]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DD, exist_ok=True)

# Rutas de salida
ruta_seccion = os.path.join(DATA_OUTPUTS_DD, "DD_seccion.csv")
ruta_municipio = os.path.join(DATA_OUTPUTS_DD, "DD_municipio.csv")
ruta_provincia = os.path.join(DATA_OUTPUTS_DD, "DD_provincia.csv")

# Guardar DataFrames
DD_seccion["Seccion_id"] = DD_seccion["Seccion_id"].astype("Int64")

DD_seccion.to_csv(
    ruta_seccion,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
DD_municipio.to_csv(
    ruta_municipio,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
DD_provincia.to_csv(
    ruta_provincia,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
print(f"✅ Archivos guardados correctamente en: {DATA_OUTPUTS_DD}")

✅ Archivos guardados correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DD_Dim_demografica
